In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

In [6]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [7]:
prompt = ChatPromptTemplate.from_template(
    "다음의 이메일 내용중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

chain = prompt | llm

output = ""
for chunk in chain.stream({"email_conversation": email_conversation}):
    print(chunk.content, end="", flush=True)
    output += chunk.content

다음은 이메일의 중요한 내용입니다:

1. 발신자: 김철수 상무 (바이크코퍼레이션)
2. 수신자: 이은채 대리 (테디인터내셔널)
3. 목적: "ZENESIS" 자전거 유통 협력 논의 및 미팅 일정 제안
4. 요청 사항:
   - ZENESIS 자전거의 상세 브로슈어 요청 (기술 사양, 배터리 성능, 디자인 정보 포함)
5. 미팅 제안:
   - 일시: 1월 15일 화요일 오전 10시
   - 장소: 귀사 사무실
6. 협력 가능성에 대한 심도 있는 논의 희망

In [8]:
class EmailSummary(BaseModel):
    """이메일에서 추출한 핵심 정보"""

    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

In [9]:
structured_llm = llm.with_structured_output(EmailSummary)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Please answer the following questions in KOREAN."),
        ("human", "QUESTION:\n{question}\n\nEMAIL CONVERSATION:\n{email_conversation}"),
    ]
)

chain = prompt | structured_llm

In [11]:
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 EmailSummary 객체입니다.
response.model_dump()

{'person': '김철수',
 'email': 'chulsoo.kim@bikecorporation.me',
 'subject': '"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안',
 'summary': '바이크코퍼레이션 김철수 상무가 이은채 대리에게 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 요청과 함께 유통 협력 가능성 논의를 위한 1월 15일 오전 10시 미팅 제안.',
 'date': '2024-01-08'}

In [ ]:
# Pydantic 객체이므로 속성으로 바로 접근하거나 dict 로 변환할 수 있습니다.
print(type(response))
print(response.date)
response.model_dump()

In [12]:
chain_with_raw = prompt | llm.with_structured_output(EmailSummary, include_raw=True)

result = chain_with_raw.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

print("parsed:", result["parsed"])
print("parsing_error:", result["parsing_error"])
print("usage:", result["raw"].usage_metadata)

parsed: person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션 김철수 상무가 이은채 대리에게 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 요청과 함께 유통 협력 가능성 논의를 위한 1월 15일 오전 10시 미팅 제안함.' date='2024-01-08'
parsing_error: None
usage: {'input_tokens': 635, 'output_tokens': 123, 'total_tokens': 758, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [14]:
for i, chunk in enumerate(
    chain.stream(
        {
            "email_conversation": email_conversation,
            "question": "이메일 내용중 주요 내용을 추출해 주세요.",
        }
    )
):
    print(i, type(chunk).__name__, chunk)

0 EmailSummary person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션 김철수 상무가 이은채 대리에게 ZENESIS 자전거에 대한 상세 브로슈어 요청과 함께, 기술 사양, 배터리 성능, 디자인 정보가 필요하다고 전달함. 또한, 1월 15일 화요일 오전 10시에 미팅을 제안하여 협력 가능성을 논의하고자 함.' date='2024-01-08'


d:\jh0902\ex0915\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EmailSummary(person='김....', date='2024-01-08'), input_type=EmailSummary])
  return self.__pydantic_serializer__.to_python(
